# Code Demo — Thesis Project Overview

**Estimating Remaining Useful Life (RUL) & State of Health (SOH) of EV Batteries
via a Hybrid Physics-Informed Deep-Learning Framework**

| | |
|---|---|
| **Student** | Rajat Sharma (CH24M559) |
| **Degree** | M.Tech in Industrial Artificial Intelligence, IIT Madras |
| **Mentor** | Dr. Sandeep Patalay |

**Demo agenda:** this notebook covers the project context (overview, frameworks,
compute, data, tools, code organization, pipeline design). After this, we switch
to `Complete_Project_Notebook.ipynb` for the code walkthrough, live execution,
results, and deployment profiling.

## 1. Project Overview

### a. Problem statement
Lithium-ion batteries degrade with use. A Battery Management System (BMS) must
estimate the **State of Health (SOH)** — current capacity relative to rated
capacity — and the **Remaining Useful Life (RUL)** — cycles left until the 80%
end-of-life threshold. Getting this wrong is costly in both directions:
over-estimating health risks unsafe operation; under-estimating it retires good
batteries early.

Today's methods force a compromise:
- **Data-driven models** (LSTM, Transformer) are accurate but physically
  unconstrained — they can predict a battery "healing" itself — and the accurate
  ones are too large for BMS microcontrollers.
- **Physics models** (electrochemical PDEs) are interpretable but far too
  expensive for real-time use on embedded hardware.

### b. Objectives and application domain
**Application domain:** on-board battery prognostics for electric vehicles —
the model must ultimately fit an ARM Cortex-M4 class BMS chip
(< 1 MB flash, < 256 KB RAM, ~80 MHz).

**Proposal:** a hybrid **physics-informed GRU** — a small recurrent network whose
loss function is regularized by a battery-degradation ODE (the Verhulst law),
so it learns from data while being pushed toward physically sensible trajectories.

**Objectives:**
1. Benchmark **ten models** under a leak-free leave-one-cell-out protocol.
2. Compare a **fixed closed-form physics prior (route A)** against a
   **learnable ODE residual (route B)** — which physics mechanism works better?
3. Empirically test **homoscedastic uncertainty weighting** against a fixed
   loss weight.
4. **Measure** (not estimate) the deployment footprint: int8 quantization,
   flash, RAM, latency against the Cortex-M4 budget.

**Headline result:** the learnable-physics hybrid reaches the lowest benchmark
error (SOH RMSE 0.0292 ± 0.0100); an identical no-physics control scores 0.0377,
so the ~23% gain is attributable to the physics loss, not model capacity. The
deployed variant needs **20.2 KB** of flash — 2% of the Cortex-M4 budget.

## 2. Programming Language and Frameworks

### a. Languages used
**Python 3.12.8** — the entire pipeline (data, models, evaluation, deployment
profiling) is Python. LaTeX is used for the thesis document itself.

### b. Libraries / frameworks with versions

| Library | Version | Role |
|---|---|---|
| PyTorch | 2.5.1 | all neural models, training, autograd for the physics residual, int8 quantization |
| NumPy | 2.2.1 | numerics |
| pandas | 2.2.3 | per-cycle tables, results aggregation |
| SciPy | 1.15.1 | curve fitting (physics priors), Savitzky–Golay filter, Wilcoxon test, `.mat` parsing |
| scikit-learn | 1.6.1 | MinMax scaling, SVR baseline |
| Matplotlib | 3.10.0 | all figures (publication settings, 600 dpi) |
| Optuna | 4.9.0 | Bayesian (TPE) hyperparameter search |
| Jupyter | — | notebook environment |

No TensorFlow, LangChain, or Hugging Face — this is not an LLM project
(Section 6 of the walkthrough addresses the LLM checklist items as
not applicable).

## 3. Compute Resources

### a. CPU / GPU configuration
- **CPU:** Apple M2, 8 cores (laptop)
- **GPU:** none used — the pipeline is deliberately **CPU-only**. This is a
  design point, not a limitation: the thesis argues for models small enough
  for embedded hardware, and every model here trains in seconds to minutes
  on a CPU.

### b. RAM and storage
- **RAM:** 8 GB unified memory (peak usage during the run stays well below this)
- **Storage:** raw dataset 176 MB (NASA 71 MB + CALCE 97 MB); the notebook is
  ~2.3 MB; generated figures and tables add a few MB.

### c. Cloud / local execution environment
- **Fully local:** macOS laptop, no cloud services, no external APIs.
- A complete end-to-end run (all ten models × 4 folds × 3 seeds + ablations +
  deployment profiling) takes **20–22 minutes** on this machine, measured on
  the two most recent full runs.

## 4. Dataset Details

### a. Dataset source and size
Two public run-to-failure datasets, both LiCoO₂ (LCO) chemistry:

| Dataset | Cells | Form | Rated | Records |
|---|---|---|---|---|
| **NASA PCoE** (Prognostics Center of Excellence) | B0005, B0006, B0007, B0018 | 18650 cylindrical | 2.0 Ah | raw `.mat` files + per-cycle CSVs; tens of thousands of per-second points per cell |
| **CALCE** (Univ. of Maryland) | CS2_35 | prismatic | 1.1 Ah | **260,931** raw channel records across date-split CSV files |

### b. Number of samples / classes / tokens
This is a **regression** task — no classes, no tokens.
- **Labels:** per-cycle SOH (capacity ÷ rated) and RUL derived from the 80%
  end-of-life threshold with a 5-cycle persistence rule.
- **Samples:** NASA 168/168/168/132 = **636 labeled discharge cycles**;
  CALCE **878 cycles** (1.138 → 0.302 Ah over its full life).
- After sliding-window tensorization (window = 10 cycles): ~600 NASA windows
  and 869 CALCE windows, each of shape 10 × 9 (8 features + cycle coordinate).

### c. Preprocessing and train–test split
**Preprocessing** (all shown live in the walkthrough):
- CALCE assembly: files ordered by internal timestamp, cycle-index offsets
  accumulated across files, one byte-identical duplicate file dropped
  (MD5-confirmed), reference-cycle outliers filtered to a plausible
  [0.30, 1.30] Ah band.
- NASA: charge/discharge operations paired by raw sequence order (the raw
  files contain extra corrupted charge logs), corrupted charge cycles flagged.
- Features: incremental-capacity analysis (dQ/dV peak voltage, magnitude,
  area) with Savitzky–Golay smoothing, plus discharge statistics and an
  internal-resistance proxy — 8 features per cycle.
- Imputation strictly within each cell; **MinMax scaler fit on training cells
  only** (no leakage).

**Train–test split — leave-one-cell-out (LOCO):** 4 folds; in each fold one
whole cell is the test set, another whole cell is validation (early stopping),
and the remaining two are training. Whole cells never mix across splits. Each
fold runs with 3 random seeds. A separate **zero-shot transfer** trains on all
four NASA cells and tests on CALCE.

## 5. Tools and Execution Environment

| Aspect | Choice |
|---|---|
| ML framework | PyTorch 2.5.1 (CPU) |
| Hardware | Apple M2 laptop, 8-core, 8 GB — local, no cloud |
| HPO | Optuna TPE (Bayesian) on a development fold |
| Quantization | PyTorch dynamic int8, `qnnpack` engine (sizes **measured** from serialized models) |
| Export | ONNX (opset 13) — `hybrid_B_deployed.onnx`, 49.8 KB |
| Statistics | SciPy paired Wilcoxon + bootstrap confidence intervals |
| LLM frameworks | none — not applicable to this project |

## 6. Code Organization

### a. Modular or single-file implementation
A **single, self-contained notebook**: `Complete_Project_Notebook.ipynb`
(59 cells). Every function is defined inside it — there are no hidden helper
modules — so an examiner can reproduce every number from one file plus the raw
data. Within the notebook the code **is modular**: labels, parsers, features,
priors, models, trainer, metrics, and profiler are separate, reusable functions
and classes.

### b. Notebook-based or production-style structure
Notebook-based by design (this is a research artifact), but with
production-style habits:
- a single `CONFIG` dictionary replaces scattered constants (acts as the
  configuration file);
- every model is seeded **immediately before construction**, so results are
  independent of execution order;
- automated data-invariant checks (`assert`s) lock the cleaned data to the
  verified ground truth;
- every figure and table is auto-saved to `figures/` (600 dpi PNG + PDF) and
  `tables/` (CSV + LaTeX) with stable names — the thesis tables come straight
  from these files.

### c. Folder hierarchy and configuration files
```
phase 3/
├── code/
│   ├── Complete_Project_Notebook.ipynb   ← everything runs from here
│   ├── figures/                          ← auto-saved (fig_01 … fig_28)
│   ├── tables/                           ← auto-saved (tbl_01 … tbl_20)
│   └── hybrid_B_deployed.onnx            ← deployment export
├── data/
│   ├── NASA/    (cycle summaries, discharge timeseries, raw data/*.mat)
│   └── CALCE/   (CS2_35 channel CSV exports)
├── 1-frontmatter/, 2-mainmatter/, zz-imp/  ← thesis LaTeX sources
└── README.md                              ← standalone project documentation
```
The notebook auto-locates `data/` by searching upward from its own folder —
no hard-coded paths.

## 7. Pipeline Design

### a. Automatic or semi-automatic workflow
**Fully automatic:** run the notebook top-to-bottom and it rebuilds everything
from the raw measurements — no manual steps in between. One flag
(`QUICK_DEMO = True`) switches to a single-seed fast mode; the canonical
3-seed numbers are printed alongside for reference either way.

### b. Data flow
```
raw measurements (.mat / .csv)
        │
        ▼
Phase 1  Data foundation      cleaning, dedup, outlier filter,
                              SOH / EOL / RUL labels, invariant checks
        │
        ▼
Phase 2  Features & physics   ICA (dQ/dV) + discharge features,
                              closed-form prior fits (logistic, double-exp, AIC),
                              leak-free windowing (train-only scaler)
        │
        ▼
Phase 3  Models               GRU, LSTM, 2× Transformer, Neural-ODE, SVR,
                              curve fit, no-physics control, hybrid A / B
        │
        ▼
Phase 4  Evaluation           4-fold LOCO × 3 seeds, cross-dataset transfer,
                              PHM metrics, paired statistics, ablations
        │
        ▼
Phase 5  Deployment           int8 quantization, MAC / RAM / latency profiling
                              vs. Cortex-M4 budget, ONNX export
```
Training happens **offline** (this pipeline); the deployed artifact is the
small quantized inference network only.

---
*Next: switch to `Complete_Project_Notebook.ipynb` for the code walkthrough —
important functions, live execution, results, performance metrics, and the
deployment profile.*